In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.common.exceptions import ElementClickInterceptedException
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
import pandas as pd
import time
import random
# CONFIG
HOTEL_NAME = "Parkroyal Collection Marina Bay Singapore "
PLATFORM = "Agoda"
URL = "https://www.agoda.com/marina-mandarin-singapore-hotel/hotel/singapore-sg.html"
OUTPUT_FILE = "test2.csv"
MAX_PAGES = 2 

# HELPERS
def human_delay(min_sec=2, max_sec=4): # adds a random delay to mimic human browsing behavior and avoid detection as a bot
    time.sleep(random.uniform(min_sec, max_sec))

def scroll_to_bottom(driver): # scrolls the page down to trigger lazy-loaded content 
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    human_delay(1,2)

def hide_overlays(driver): # removes blocking UI elements that might interfere with clicking buttons
    try:
        driver.execute_script(""" 
            const el = document.querySelector('[data-selenium="checkInBox"]');
            if (el) { el.style.display='none'; }
        """)
    except:
        pass
    human_delay(0.5,1)

# SETUP SELENIUM
options = Options()
options.headless = False
options.add_argument("--start-maximized")
options.add_argument("--disable-blink-features=AutomationControlled")

service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=options)
driver.get(URL)
human_delay(5,7)

# SCRAPE REVIEWS
reviews_data = []   
current_page = 1

while current_page <= MAX_PAGES:
    print(f"Scraping page {current_page}...")

    scroll_to_bottom(driver)
    hide_overlays(driver)
    human_delay(2,3)

    soup = BeautifulSoup(driver.page_source, "html.parser")
    review_cards = soup.select("div.Review-comment")

    if not review_cards:
        print("No reviews found.")
        break

    for r in review_cards:
        try: title = r.select_one("h4[data-testid='review-title']").get_text(strip=True)
        except: title = None
        try: rating = r.select_one("div.Review-comment-leftScore").get_text(strip=True)
        except: rating = None
        try: review_text = r.select_one("p.Review-comment-bodyText").get_text(strip=True)
        except: review_text = None
        try: guest_type = r.select_one("div[data-info-type='group-name'] span").get_text(strip=True)
        except: guest_type = None

        reviews_data.append({
            "title": title,
            "rating": rating,
            "review_text": review_text,
            "guest_type": guest_type
        })

    # Attempt to click next page
    try:
        next_btn = driver.find_element(By.XPATH, f'//button[normalize-space()="{current_page+1}"]')
        driver.execute_script("arguments[0].scrollIntoView({block:'center'});", next_btn)
        human_delay(1,2)
        driver.execute_script("arguments[0].click();", next_btn)
        human_delay(5,7)
        current_page += 1
    except ElementClickInterceptedException:
        print("Click intercepted, retrying...")
        scroll_to_bottom(driver)
        hide_overlays(driver)
        human_delay(2,3)
    except:
        print("Could not click next page. Stopping.")
        break

# SAVE TO CSV
driver.quit()
df = pd.DataFrame(reviews_data)
df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

print(f"Scraping complete. Total reviews: {len(df)}")
print(df.head(5))


Scraping page 1...
Scraping page 2...
Scraping complete. Total reviews: 10
                                               title rating  \
0     “Modern Design, Great Vibes, Perfect Location”   10.0   
1  “Excellent location - but could do with a few ...    9.2   
2                                   “Good ambience ”   10.0   
3                                    “Memorial stay”   10.0   
4                                   “Memorable stay”    8.8   

                                         review_text  \
0  Our stay at PARKROYAL COLLECTION Marina Bay, S...   
1  Park royal collection marina bay is located at...   
2  I recently enjoyed a delightful one-night stay...   
3  Review for PARKROYAL COLLECTION Marina Bay\n\n...   
4  We had a lovely stay at Parkroyal Marina Bay, ...   

                   guest_type  
0                      Couple  
1                       Group  
2               Solo traveler  
3                      Couple  
4  Family with young children  


The website was changed to Agoda.com because the initial website booking.com had explicitly disallowed scraping on their website.